In [ ]:
class Generator(nn.Module):
    def __init__(self, input_noise_dim=64, output_dim=784):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_noise_dim, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, output_dim),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.model(x)
        return x


class Discriminator(nn.Module):
    def __init__(self, input_dim=784):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        x = self.model(x)
        return x


transform = v2.ToTensor()
mnist_train_dataset = torchvision.datasets.MNIST(root='./', download=True, train=True, transform=transform)
mnist_test_dataset = torchvision.datasets.MNIST(root='./', download=True, train=False, transform=transform)

train_loader = DataLoader(mnist_train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(mnist_test_dataset, batch_size=64, shuffle=False)
noise_dim = 64
generator = Generator(input_noise_dim=noise_dim, output_dim=784).to(device)
discriminator = Discriminator(input_dim=784).to(device)

criterion = nn.BCELoss()
optim_generator = torch.optim.Adam(generator.parameters(), lr=1e-4)
optim_discriminator = torch.optim.Adam(discriminator.parameters(), lr=1e-4)
for epoch in range(10):
    for real_imgs, _ in tqdm(train_loader):
        real_imgs = real_imgs.to(device)
        real_imgs = real_imgs.view(real_imgs.shape[0], -1)
        batch_size = real_imgs.shape[0]

        # Train Discriminator
        optim_discriminator.zero_grad()

        noise = torch.randn(batch_size, noise_dim).to(device)
        fake_imgs = generator(noise)

        real_labels = torch.ones(batch_size, 1).to(device)
        fake_labels = torch.zeros(batch_size, 1).to(device)

        real_preds = discriminator(real_imgs)
        fake_preds = discriminator(fake_imgs)

        real_loss = criterion(real_preds, real_labels)
        fake_loss = criterion(fake_preds, fake_labels)

        loss_dicriminator = (real_loss + fake_loss) / 2
        loss_dicriminator.backward()
        optim_discriminator.step()

        # Train Generator
        optim_generator.zero_grad()
        noise = torch.randn(batch_size, noise_dim).to(device)
        fake_imgs = generator(noise)
        fake_preds = discriminator(fake_imgs)

        loss_generator = criterion(fake_preds, real_labels)
        loss_generator.backward()
        optim_generator.step()

    print(f"epoch: {epoch}, Loss D: {loss_dicriminator.item()}, loss G: {loss_generator.item()}")
noise = torch.randn(1, noise_dim).to(device)
fake_imgs = generator(noise)
fake_imgs = fake_imgs.view(fake_imgs.shape[0], 1, 28, 28)
noise
plt.imshow(fake_imgs.cpu().detach().squeeze(), cmap='gray')
# 2025-07-23
## Conditional GAN
import torch
